## Optimise Prompt

### Packages

In [1]:
# Packages
import os

import mlflow
from dotenv import load_dotenv
from mlflow.genai.optimize import MetaPromptOptimizer
from openai import AzureOpenAI

# Load environment variables
load_dotenv()

# Set MLflow tracking URI
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

# Rename some environmental variables for convenience
os.environ["AZURE_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"] = os.environ["AZURE_OPENAI_ENDPOINT"]
os.environ["AZURE_API_VERSION"] = os.environ["AZURE_OPENAI_API_VERSION"]

### Optimise Prompt

In [4]:
def get_client() -> AzureOpenAI:
    return AzureOpenAI(
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    )


# Define your prediction function
def predict_fn(prompt_url: str, **kwargs) -> str:
    client = get_client()
    prompt = mlflow.genai.load_prompt(prompt_url)
    completion = client.chat.completions.create(
        model="gpt-5-mini",  # Replace with your model
        messages=[{"role": "user", "content": prompt.format(**kwargs)}],
    )
    return completion.choices[0].message.content


# Optimise your prompt
def optimise_prompt(prompt_url: str):
    return mlflow.genai.optimize_prompts(
        predict_fn=lambda **kwargs: predict_fn(prompt_url=prompt_url, **kwargs),
        train_data=[],
        prompt_uris=[prompt_url],
        optimizer=MetaPromptOptimizer(
            reflection_model="azure:/gpt-5.6-terra",
            guidelines=(
                "This prompt is used in a reporting agent that generates reports describing "
                "the state of the coast. The optimized prompt must be returned exclusively as "
                "a JSON-serialized list of chat message dictionaries, for example: "
                '[{"role": "system", "content": "..."}]. Do not return plain text, Markdown, '
                "code fences, or any explanation."
            ),
        ),
        scorers=[],
    )


result = optimise_prompt("prompts:/sotc.rag.memo/2")

# Retrieve and print the optimised prompt
optimised_prompt = result.optimized_prompts[0]
print(f"Optimised template: {optimised_prompt.template}")

2026/09/24 14:00:45 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: No training data provided, using zero-shot metaprompting
2026/09/24 14:00:45 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: Applying zero-shot prompt optimization with best practices
2026/09/24 14:01:01 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: Successfully generated improved prompts
2026/09/24 14:01:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: sotc.rag.memo, version 4


🏃 View run gaudy-calf-248 at: http://108.141.0.183/mlflow/#/experiments/0/runs/9555dfc6321642a38552a3e689ce4a1f
🧪 View experiment at: http://108.141.0.183/mlflow/#/experiments/0
Optimised template: [{'role': 'system', 'content': 'You are an expert coastal-state reporting analyst. Generate a precise, evidence-based memo describing the state of the coast for the area(s) of interest specified in the supplied material.\n\n## Source-bound requirements\n1. Treat the supplied figures and datasets as the complete evidence base. Use only information explicitly supported by them.\n2. Do not invent, estimate, extrapolate, calculate, infer causation, or add background knowledge. If the evidence does not support a claim, omit the claim.\n3. Datasets provide context for interpretation but must never be named, cited, or referenced in the memo.\n4. Distinguish direct observations from comparisons. Use cautious wording such as "shows," "indicates," or "is higher/lower than" only when directly supported